In [ ]:
# Kaggle kernel — Monarch film chunk (auto-generated by tools/make_kaggle_kernel.py)
# Har GPU par alag worker (Kaggle par T4 x2 milta hai) taake waqt aadha lage.
import json, os, pathlib, shutil, subprocess, sys

BRANCH = "arena/3d8cfad7-monarch-studio"
REPO = "https://github.com/adil-chandio/Monarch-Studio"
ONLY = "k01"
JOB = ""

repo = pathlib.Path("/kaggle/working/repo")
if not (repo / "tools/kaggle_stick_film.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, str(repo)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers>=0.32",
                "transformers>=4.49", "accelerate", "ftfy", "sentencepiece", "safetensors",
                "regex", "imageio-ffmpeg"], check=False)
print("deps installed", flush=True)

scenes = [s for s in ONLY.split(",") if s.strip()]
try:
    import torch
    n_gpu = torch.cuda.device_count()
    for i in range(n_gpu):
        print("GPU", i, torch.cuda.get_device_name(i), flush=True)
except Exception as exc:
    print("torch check fail:", exc, flush=True)
    n_gpu = 1

groups = [scenes]
if n_gpu >= 2 and len(scenes) > 1:
    half = (len(scenes) + 1) // 2
    groups = [scenes[:half], scenes[half:]]
    print("parallel workers:", groups, flush=True)

procs = []
for i, group in enumerate(groups):
    out = f"/kaggle/working/stick-w{i}"
    cmd = [sys.executable, str(repo / "tools/kaggle_stick_film.py"),
           "--out", out, "--no-kit", "--only", ",".join(group)]
    if JOB:
        cmd += ["--job", JOB]
    env = dict(os.environ)
    if len(groups) > 1:
        env["CUDA_VISIBLE_DEVICES"] = str(i)
    print("$", " ".join(cmd), "| CUDA_VISIBLE_DEVICES=" + env.get("CUDA_VISIBLE_DEVICES", "all"), flush=True)
    procs.append(subprocess.Popen(cmd, env=env))

codes = [p.wait() for p in procs]
print("worker exit codes:", codes, flush=True)

clips = pathlib.Path("/kaggle/working/clips"); clips.mkdir(exist_ok=True)
for i in range(len(groups)):
    for f in pathlib.Path(f"/kaggle/working/stick-w{i}").glob("*.mp4"):
        if f.stat().st_size > 20000:
            shutil.copy2(f, clips / f.name)
got = sorted(p.name for p in clips.glob("*.mp4"))
print("CLIPS:", got, flush=True)
print(json.dumps({"only": ONLY, "clips": got, "exit": codes}))
